# Transaction labeling

On classe une transaction carte dans une catégorie à partir du nom du marchand. Deux niveaux d'étiquette existent (`minor_category`, 12 classes, et `major_category`, 8 classes), mais `minor` détermine `major` de façon unique : je prédis donc seulement `minor_category` et je déduis `major` par table.

Approche : nettoyer le nom (dont le bruit des processeurs de paiement), TF-IDF au niveau caractère, réduction SVD, RandomForest. Le modèle est sauvegardé dans `model.joblib`, que l'API (`app.py`) charge ensuite.

In [ ]:
import re
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_csv('data/transactions.csv')
df['created_at'] = pd.to_datetime(df['created_at'])
df.shape

In [ ]:
df.head()

## Vérifications sur les données

In [ ]:
print('transaction_id tous uniques :', df['transaction_id'].is_unique)
id_to_name = df.groupby('merchant_id')['merchant_name'].nunique()
name_to_id = df.groupby('merchant_name')['merchant_id'].nunique()
print('merchant_id -> 1 seul nom  :', round((id_to_name <= 1).mean(), 3))
print('merchant_name -> 1 seul id :', round((name_to_id <= 1).mean(), 3))

In [ ]:
pd.DataFrame({'nulls': df.isna().sum(), 'nunique': df.nunique()})

Constats : `transaction_id` unique ; un même `merchant_name` recouvre souvent plusieurs `merchant_id` (plusieurs points de vente) ; `amount_currency`, `card_type`, `pfm_recurring_frequency` sont constantes ou quasi vides (ignorées) ; `title` vaut `merchant_name` mais sans trous, donc je garde `title`.

### Pourquoi des montants positifs et négatifs ?

In [ ]:
df.groupby('minor_category')['amount_value'].agg(['min', 'max', 'mean']).round(2)

Seul `refund` est positif (un remboursement est un crédit). Le signe du montant est donc un signal direct du remboursement, gardé comme feature.

### Distribution des deux cibles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['minor_category'].value_counts().plot(kind='barh', ax=axes[0])
axes[0].set_title('minor_category')
df['major_category'].value_counts(dropna=False).plot(kind='barh', ax=axes[1])
axes[1].set_title('major_category')
plt.tight_layout()
plt.show()

Classes déséquilibrées : j'en tiens compte avec `class_weight` et en évaluant en macro-F1.

### Correspondance minor -> major

In [ ]:
check = df.dropna(subset=['major_category']).groupby('minor_category')['major_category'].nunique()
print('chaque minor a un seul major :', (check == 1).all())
mapping = (df.dropna(subset=['major_category'])
             .groupby('minor_category')['major_category']
             .agg(lambda s: s.mode().iat[0]))
MINOR_TO_MAJOR = mapping.to_dict()
MINOR_TO_MAJOR['refund'] = None
MINOR_TO_MAJOR

## Nettoyage du texte
Les descripteurs portent du bruit, notamment des préfixes de processeurs de paiement (`PAYPAL *`, `SQ *`...) qui masquent le vrai marchand. Je les retire avant de vectoriser.

In [ ]:
def clean_text(x):
    x = str(x).lower()
    x = re.sub(r'\b(paypal|sq|sumup|ztl|zettle|stripe)\s*\*', ' ', x)
    x = re.sub(r'[^a-z\s]', ' ', x)
    return re.sub(r'\s+', ' ', x).strip()

clean_text('PAYPAL *UBER BV')

## Features
Le nom nettoyé plus quelques signaux numériques (remboursement), et un drapeau `amount_isnan`.

In [ ]:
def make_features(d):
    d = d.copy()
    d['clean_title'] = d['title'].fillna('').map(clean_text)
    amt_raw = pd.to_numeric(d['amount_value'], errors='coerce')
    d['amount_isnan'] = amt_raw.isna().astype(int)
    amt = amt_raw.fillna(0)
    d['is_credit'] = (amt > 0).astype(int)
    status = d['status'] if 'status' in d else pd.Series(['completed'] * len(d))
    d['is_refunded'] = (status.astype(str) == 'refunded').astype(int)
    d['log_amount'] = np.log1p(amt.abs())
    return d

df = make_features(df)
df[['clean_title', 'is_credit', 'is_refunded', 'log_amount', 'amount_isnan']].head()

## Le modèle
TF-IDF **caractère** (`char_wb` 3-5, robuste aux noms collés/tronqués) -> SVD pour densifier -> RandomForest. Je borne `max_depth=16` : ça limite le surapprentissage et garde l'artefact léger (~28 Mo, committable).

In [ ]:
text_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, analyzer='char_wb', ngram_range=(3, 5))),
    ('svd', TruncatedSVD(n_components=200, random_state=42)),
])
num_cols = ['is_credit', 'is_refunded', 'log_amount', 'amount_isnan']
pre = ColumnTransformer([
    ('text', text_pipe, 'clean_title'),
    ('num', 'passthrough', num_cols),
])
model = Pipeline([
    ('pre', pre),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=16,
                                  class_weight='balanced_subsample',
                                  random_state=42, n_jobs=-1)),
])

## Évaluation
Split temporel : apprentissage sur les 80 % de transactions les plus anciennes, test sur les 20 % les plus récentes.

In [ ]:
d = df.sort_values('created_at').reset_index(drop=True)
y = d['minor_category']
cut = int(len(d) * 0.8)
model.fit(d.iloc[:cut], y.iloc[:cut])
pred = model.predict(d.iloc[cut:])
print('accuracy :', round(accuracy_score(y.iloc[cut:], pred), 3))
print('macro-F1 :', round(f1_score(y.iloc[cut:], pred, average='macro'), 3))

In [ ]:
print(classification_report(y.iloc[cut:], pred, zero_division=0))

Macro-F1 plus parlante que l'accuracy vu le déséquilibre. Les classes sans marchand récurrent (others_shopping) restent les plus faibles.

In [ ]:
pd.crosstab(y.iloc[cut:], pred, normalize='index').round(2)

## Modèle final et sauvegarde
Je réentraîne sur toutes les données et je sauvegarde le modèle **et** la table `minor -> major` dans `model.joblib`. C'est ce fichier que l'API charge.

In [ ]:
final_model = model.fit(d, y)
joblib.dump({'model': final_model, 'minor_to_major': MINOR_TO_MAJOR}, 'model.joblib')
print('modele sauvegarde dans model.joblib')

## Prédire une transaction
Même logique que dans `app.py`.

In [ ]:
def predict_one(txn):
    row = make_features(pd.DataFrame([txn]))
    minor = final_model.predict(row)[0]
    proba = float(final_model.predict_proba(row)[0].max())
    return {'minor_category': minor,
            'major_category': MINOR_TO_MAJOR.get(minor),
            'confidence': round(proba, 3)}

for t in [
    {'title': 'CARREFOURMARKET', 'amount_value': -12.5},
    {'title': 'BETCLIC', 'amount_value': -50},
    {'title': 'RETRAIT DAB', 'amount_value': -50},
    {'title': 'REMBOURSEMENT', 'amount_value': 29.9, 'status': 'refunded'},
    {'title': 'UN MARCHAND INCONNU XYZ', 'amount_value': -9.9},
]:
    print(t['title'], '->', predict_one(t))

## L'API
Le service est dans `app.py`. Il **charge** `model.joblib` (créé ci-dessus) et expose `POST /predict`. Si le fichier est absent, il réentraîne automatiquement.

```bash
uvicorn app:app --port 8000
# puis ouvrir http://127.0.0.1:8000/docs
```

## Si j'avais eu plus de temps

- Comparer à un baseline plus simple (régression logistique sur TF-IDF) pour isoler l'apport de la SVD et du RandomForest.
- Évaluer aussi sur un split par marchand (`GroupShuffleSplit`) pour mesurer la généralisation aux marchands inconnus, pas seulement dans le temps.
- **Enrichir les marchands inconnus via une API externe** : pour un marchand jamais vu, le nom seul ne suffit pas. Je récupérerais une courte description du marchand via une API de recherche (Tavily / Google Places, ou le répertoire SIRENE pour l'activité), puis je calculerais un **embedding** de cette description (via une API d'embeddings) que j'ajouterais comme feature. Ça attaque la longue traîne de marchands inconnus, là où le TF-IDF sur le nom plafonne.
- Traiter les cas particuliers par des règles : `refund` via le `status`, l'ATM via une liste de banques / mots-clés.
- En production, utiliser le code MCC de la transaction (catégorie marchand standard de l'industrie), absent de ce jeu synthétique mais le signal le plus fiable. Côté texte, un `HashingVectorizer` (sans vocabulaire à persister) simplifierait le service à grande échelle.
- Renvoyer une catégorie `incertain` sous un seuil de confiance, pour qu'une appli PFM demande confirmation à l'utilisateur au lieu de mal classer.